# 37. The city average that no order adds up to

**Track:** debug-the-ai | **Difficulty:** easy | **Tags:** aggregation, average of averages, weighted average, group by

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("037-average-of-averages")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "orders": """
UEFSMRUEFdAiFcYRTBWqBBUAEgAAqBEIMXUABQEAMg0IADMNCAA0DQgANQ0IADYNCAA3DQgAOA0IADkNCAA6DQgAOw0IADwNCAA9
DQgAPg0IAD8NCABADQgAQQ0IAEINCABDDQgARA0IAEUNCABGDQgARw0IAEgNCABJDQgASg0IAEsNCABMDQgATQ0IAE4NCABPDQgA
UA0IAFENCABSDQgAUw0IAFQNCABVDQgAVg0IAFcNCABYDQgAWQ0IAFoNCABbDQgAXA0IAF0NCABeDQgAXw0IAGANCABhDQgAYg0I
AGMNCABkDQgAZQ0IAGYNCABnDQgAaA0IAGkNCABqDQgAaw0IAGwNCABtDQgAbg0IAG8NCABwDQgAcQ0IAHINCABzDQgAdA0IAHUN
CAB2DQgAdw0IAHgNCAB5DQgAeg0IAHsNCAB8DQgAfQ0IAH4NCAB/DQgAgA0IAIENCACCDQgAgw0IAIQNCACFDQgAhg0IAIcNCACI
DQgAiQ0IAIoNCACLDQgAjA0IAI0NCACODQgAjw0IAJANCACRDQgAkg0IAJMNCACUDQgAlQ0IAJYNCACXDQgAmA0IAJkNCACaDQgA
mw0IAJwNCACdDQgAng0IAJ8NCACgDQgAoQ0IAKINCACjDQgApA0IAKUNCACmDQgApw0IAKgNCACpDQgAqg0IAKsNCACsDQgArQ0I
AK4NCACvDQgAsA0IALENCACyDQgAsw0IALQNCAC1DQgAtg0IALcNCAC4DQgAuQ0IALoNCAC7DQgAvA0IAL0NCAC+DQgAvw0IAMAN
CADBDQgAwg0IAMMNCADEDQgAxQ0IAMYNCADHDQgAyA0IAMkNCADKDQgAyw0IAMwNCADNDQgAzg0IAM8NCADQDQgA0Q0IANINCADT
DQgA1A0IANUNCADWDQgA1w0IANgNCADZDQgA2g0IANsNCADcDQgA3Q0IAN4NCADfDQgA4A0IAOENCADiDQgA4w0IAOQNCADlDQgA
5g0IAOcNCADoDQgA6Q0IAOoNCADrDQgA7A0IAO0NCADuDQgA7w0IAPANCADxDQgA8g0IAPMNCAD0DQgA9Q0IAPYNCAD3DQgA+A0I
APkNCAD6DQgA+w0IAPwNCAD9DQgA/g0IAP8NCAQAdsV3BAABDQgAAg0IAAMNCAAEDQgABQ0IAAYNCAAHDQgACA0IAAkNCAAKDQgA
Cw0IAAwNCAANDQgADg0IAA8NCAAQDQgAEQ0IABINCAATDQgAFA0IABUNCAAWDQgAFw0IABgNCAAZDQgAGg0IABsNCAAcDQgAHQ0I
AB4NCAAfDQgAIA0IACENCAAiDQgAIw0IACQNCAAlDQgAJg0IACcNCAAoDQgAKQ0IACoNCAArDQgALA0IAC0NCAAuDQgALw0IADAN
CAAxDQgAMg0IADMNCAA0DQgANQ0IADYNCAA3DQgAOA0IADkNCAA6DQgAOw0IADwNCAA9DQgAPg0IAD8NCABADQgAQQ0IAEINCABD
DQg8RHYAAAAAAABFdgAAAAAAABUAFYgFFZIFLBWqBBUQFQYVBhwYCEV2AAAAAAAAGAgxdQAAAAAAABYAKAhFdgAAAAAAABgIMXUA
AAAAAAAREQAAAMQC9EMBAwAAAKoEAQlHAAIIGECggIEDCBIoWMCggYMHECJImEChgoULGDJo2MChg4cPIEKIGEGihIkTKFKoWMGi
hYsXMGLImEGjho0bOHLo2MGjh48fQIIIGUKkiJEjSJIoWcKkiZMnUKJImUKlipUrWLJo2cKli5cvYMKIGUOmjJkzaNKoWcOmjZs3
cOLImUOnjp07ePLo2cOnj58/gAIJGkSokKFDiBIpWsSokaNHkCJJmkSpkqVLmDJp2sSpk6dPoEKJGkWqlKlTqFKpWsWqlatXsGLJ
mkWrlq1buHLp2sWrl69fwIIJG0asmLFjyJIpW8asmbNn0KJJm0atmrVr2LJp28atm7dv4MKJG0eunLlz6NKpW8eunbt38OLJm0ev
nr17+PLp28evn79/AAMKHEiwoMGDCBMqXMiwocOHECNKnEgRAAAAFQQV8AEVkAFMFR4VABIAAHgEgwAJAQBlCQcEAI0NCABvDQgA
cQ0IAJcNCACEDQgAjg0IAHANCAB7DQgAZg0IAHoNCABnDQg8eQAAAAAAAABoAAAAAAAAABUAFaoCFbICLBWqBBUQFQYVBhwYCJcA
AAAAAAAAGAhlAAAAAAAAABYAKAiXAAAAAAAAABgIZQAAAAAAAAAREQAAAJUB8JQDAAAAqgQBBEcQIANANVMWcJgzVwMZGQMRGgEz
MLF5G7kcMqNbNhq5QgETtDuzF52hORETJxubEbuRCw0BuxkFEzO+cRcRWJARGzcRl7s3M7uRyxmx0eExkRJcvTVR0TsRockDMDvT
C7N7UhMeMZA7Cyk4qRUdy7symlUYAxgzuzspAxwCCqObACY94JYQArEXCbUBABUEFXIVcEwVDBUAEgAAOUgHAAAAQ2hlbm5haQYA
AABNdW1iCQp8SW5kb3JlBAAAAFB1bmUFAAAAS29jaGkFAAAARGVsaGkVABXkARXqASwVqgQVEBUGFQYcNgAoBFB1bmUYB0NoZW5u
YWkREQAAAHLwcQMAAACqBAEDRwg0YNyIQOsmDk0zJEmwYWnVtIm2lFjSasGydqvSJl2ySE2btGlRBG1DLFsaKcmIJo2WqK22tWnT
pGmSpUrYcmHaJUkzYN1arJW4JBnaFdXWME3Taim5DMvWdtWQCMHSAlCHoIiQKgUbABUEFaAVFZYVTBWgBBUAEgAA0Ar0SAEAAAJZ
LAAABHxvAAACbFAAAAE27wAAA0g8AAABPyQAAALEAgAAAPmcAAABkrwAAAInLAAAAx5wAAAAqFwAAAC92AAAA0RUAAAB5VoAAAMK
IAAAAGz8AAAA844AAANTLAAAAsaLAAAB9+cAAAGgNgAAAg1jAAAC748AAAFSrAAABIzYAAAB0zAAAALRfAAAA7NDAAABxkwAAANj
xgAAAv8wAAAAT0wAAAOVxgAAA3isAAACYDMAAAMFDAAAAy1IAAAC6CQAAALVAAAAA8IcAAABJb8AAAF+ngAAAz8OAAABWuAAAAJP
aAAAAb92AAABBhsAAABpEwAAAy7YAAABDesAAAJPAwAAAhqrAAAAsYoAAACeMwAAAU0CAAAA7kcAAALMBAAAANbYAAAFzrgAAAFT
2AAAARmkAAAAuGAAAAEcxAAABAn8AAACrgEeZHx4AAAFaVoAAAAPoAAAAFqgAAABFYoAAAMWAQ/wqvdEAAACNEIAAATgogAABA4W
AAABD64AAALscAAAAOL0AAABf/sAAALLCgAAAxhiAAADgRIAAAM/owAAA/CYAAADKcQAAAHOfwAAAL/MAAADkOQAAABlXgAAAUFK
AAACE0AAAAQ1vAAAAV/0AAAAtgcAAASe0AAAAGI+AAAAvaYAAALWLAAAAvSkAAAB5xwAAAK/hAAAAs5bAAABHiIAAAD15gAAAZ4Q
AAABwyHlkDGTAAABOtgAAAMj6AAAAs34AAAB5+QAAAB71AAAAJpLAAACbB4B3BA8AAACpEFEFODUAAAD3gG+VFcUAAAArjcAAAH9
YAAAAeyUAAABU3Mh0fBplAAAAwuvAAAA3XwAAAG0hgAAAqYfAAAELewAAAN9WwAAAzwgAAAB1h4AAADhAAAAANegAAABCgMAAAL2
ygAAAvVsAAACHmIAAAQCkAAAARh4AAABGUAAAAMlRgAAAKnrAAAA0vAAAAC7gAF4EPgAAANlIeUotiQAAAEXTAAABLUhpCjNWgAA
BW6fAAAAdwF4eKIUAAACIbQAAAMkGgAAAO0cAAABwf8AAAUy3AAAAIohqWS32AAABmy6AAAAxtMAAAEsxwAAAilSAAAEagEj4OCc
AAAEMXAAAAaHGAAAAPE2AAAC78IAAASkegAAAtBQAAADotwAAACyIAAAAP4aAAAAOZ4AAAOqSCFZ9MwBYgAAAkJSAAABd2MAAADc
tAAABAouAAADdxsAAAKROgAAAX1yAAAB0NgAAAMc4AAAAPRWAAABRiwAAALh5AAAAWN3AAABjtQAAAIITwAABdAWAAAAXiQAAAN2
UwAABCPEAAACvlcAAAM6LAAAAZgCAAAA39MAAAPuQAAAAWNGAAACyOMAAACYVwAAAMcGAAACLc8AAAA7YAAAAPkGAAABicAAAAKN
5wAAAuQ8AAAD9hAAAADgzgAAAFhIAAAAsugAAADKWAAAAXUMAAABSOgAAACkdAAAAel0AAACPwAAAACbFAAABAjPAAACr+QAAAHQ
EAAAAFgWAAADAewAAAMRjAAAA5ZbAAAAw4IAAAEULAAAAGs6AAABuWgAAAENVgAAAd7oAAACdngAAAGbtwAAAInkAAAFz+QAAADS
7wAAAJYyAAAA2lwAAAIVlwAAAGsIAAAAwPcAAAEDXwAAAjG3AAABk+cAAADOPwAAAeeAAAADlyQAAAFxJAAAAXVvAAAArEQAAACv
YwAAAYfLAAAC3AcAAASPLwAAAN0YAAACTjwAAAIMNwAAARfiAAAB1oIAAAKdiAAAASceAAABv6cAAACzrwAAAGBKAAADVuIVABWI
BRWSBSwVqgQVEBUGFQYcGAUAAAaHGBgFAAAAD6AWACgFAAAGhxgYBQAAAA+gEREAAADEAvRDAQMAAACqBAEJRwACCBhAoICBAwgS
KFjAoIGDBxAiSJhAoYKFCxgyaNjAoYOHDyBCiBhBooSJEyhSqFjBooWLFzBiyJhBo4aNGzhy6NjBo4ePH0CCCBlCpIiRIwKQJFGy
hEkTJ0+gRJEyhUoVK1ewZNGyhUsXL1/AhBEzhkwZM2fQpCGhZg2bNm7ewIkjZw6dOnbu4MmjZw+fPn7+AAokaBChQoYOIUqkaBGj
Ro4eQYokaRKlSpYuYcqkaROnTp4+gQolahSpUqZOoUqlahWrVq5ewYolaxatWrZu4cqlaxevXr5+AQsmbBixYsaOIUumbBmzZrCc
PYMWTdo0atWsXcOWTds2bt1gefsGLpy4ceTKmTuHLp26dezauXsHL568bfPo1bN3D18+ffv49fP3D2BAgQMJFjR4EGFChQsZNnT4
EAAAABUEGVw1ABgGc2NoZW1hFQgAFQQlAhgIb3JkZXJfaWQAFQQlAhgIc3RvcmVfaWQAFQwlAhgEY2l0eSUATBwAAAAVDhUKFQIY
BmFtb3VudCUKFQQVFCxcFQQVFAAAABaqBBkcGUwmABwVBBk1AAYQGRgIb3JkZXJfaWQVAhaqBBb+KBb+FybwESYIHBgIRXYAAAAA
AAAYCDF1AAAAAAAAFgAoCEV2AAAAAAAAGAgxdQAAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgCqBAAAACYAHBUEGTUABhAZ
GAhzdG9yZV9pZBUCFqoEFr4FFuYEJrYZJoYYHBgIlwAAAAAAAAAYCGUAAAAAAAAAFgAoCJcAAAAAAAAAGAhlAAAAAAAAABERABks
FQQVABUCABUAFRAVAgA8KQYZJgCqBAAAACYAHBUMGTUABhAZGARjaXR5FQIWqgQWxAMWyAMm+B0m7BwcNgAoBFB1bmUYB0NoZW5u
YWkREQAZLBUEFQAVAgAVABUQFQIAPBa+FxkGGSYAqgQAAAAmABwVDhk1AAYQGRgGYW1vdW50FQIWqgQWthsWthsm7DUmtCAcGAUA
AAaHGBgFAAAAD6AWACgFAAAGhxgYBQAAAA+gEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmAKoEAAAAFrZNFqoEJggW4jsAGRwYDEFS
Uk9XOnNjaGVtYRiYAy8vLy8veWdCQUFBUUFBQUFBQUFLQUF3QUJnQUZBQWdBQ2dBQUFBQUJCQUFNQUFBQUNBQUlBQUFBQkFBSUFB
QUFCQUFBQUFRQUFBQzRBQUFBY0FBQUFFQUFBQUFFQUFBQWFQLy8vd0FBQVFjUUFBQUFJQUFBQUFRQUFBQUFBQUFBQmdBQUFHRnRi
M1Z1ZEFBQUNBQU1BQVFBQ0FBSUFBQUFDZ0FBQUFJQUFBQ2cvLy8vQUFBQkJSQUFBQUFjQUFBQUJBQUFBQUFBQUFBRUFBQUFZMmww
ZVFBQUFBQUVBQVFBQkFBQUFNei8vLzhBQUFFQ0VBQUFBQndBQUFBRUFBQUFBQUFBQUFnQUFBQnpkRzl5WlY5cFpBQUFBQUM4Ly8v
L0FBQUFBVUFBQUFBUUFCUUFDQUFHQUFjQURBQUFBQkFBRUFBQUFBQUFBUUlRQUFBQUpBQUFBQVFBQUFBQUFBQUFDQUFBQUc5eVpH
VnlYMmxrQUFBQUFBZ0FEQUFJQUFjQUNBQUFBQUFBQUFGQUFBQUFBQUFBQUE9PQAYIHBhcnF1ZXQtY3BwLWFycm93IHZlcnNpb24g
MjUuMC4xGUwcAAAcAAAcAAAcAAAA4AMAAFBBUjE=
""",
}

_data_dir = pathlib.Path("data") / "037-average-of-averages"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

orders = spark.read.parquet((_data_dir / "orders.parquet").resolve().as_posix())
orders.createOrReplaceTempView("orders")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'avg_order_value': 'cbc5f9cc090a3400b3a456bf0a5350ccb62a6dcd8bacfeb0c3cdb0c758494251',
                         'city': '58e5b81477cba4579c10bf092ec43d828291519cf7654660bd35ca049b7cffd0',
                         'orders': 'f38b9416536754ed92adac7d4037efcb96a6f41f951471699138e7606d20079e'},
 'columns': [['avg_order_value', 'decimal(12,2)'], ['city', 'string'], ['orders', 'bigint']],
 'fingerprint': '7faf56323f8e0dc37f49e74fa6c5fe1faa22ae47bcf387ecb5accfcc4277e0c8',
 'order_matters': False,
 'row_count': 6}

check = make_check(EXPECTED)

## The situation

The regional dashboard shows the average order value per city. An AI assistant wrote
the query in the answer cell, reusing a per-store summary that already existed.

It runs, there is one row per city, and the order counts match the orders table
exactly. Then finance divided Mumbai's revenue by Mumbai's order count and got a
number that is nowhere near the one on the dashboard.

Find the bug and fix it.

## Input tables

### `orders`

One row per order.

| Column | Type | Description |
|---|---|---|
| `order_id` | BIGINT | Unique per order. |
| `store_id` | BIGINT | The store that took the order. |
| `city` | STRING | The city of the store. A store is in one city. |
| `amount` | DECIMAL(10,2) | The order value. Never NULL. |

## Requirement

Return one row per city.

- `orders` is the number of orders in the city.
- `avg_order_value` is the city's total order amount divided by the city's number
  of orders, rounded to 2 decimals.

## Expected output

One row per city.

| Column | Type | Description |
|---|---|---|
| `city` | STRING | |
| `orders` | BIGINT | |
| `avg_order_value` | DECIMAL(12,2) | Total amount / number of orders, rounded to 2 decimals. |

In [ ]:
orders.show(5)

## Your answer

The cell below is the AI-written code. It runs, but its output is wrong. Find the bug and fix it. The tables are available as DataFrames and as views: `orders`.

In [ ]:
query = """
-- Average order value per city, built from the per-store statistics
WITH store_stats AS (
    SELECT
        city,
        store_id,
        COUNT(*) AS orders,
        AVG(amount) AS avg_amount
    FROM orders
    GROUP BY city, store_id
)
SELECT
    city,
    SUM(orders) AS orders,
    CAST(ROUND(AVG(avg_amount), 2) AS DECIMAL(12, 2)) AS avg_order_value
FROM store_stats
GROUP BY city
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)